# Inteligencia Artificial (FP13) — **Tarea Segregación de Datpos**
## 3.4 Segregación de datos · 3.5 Modelo de entrenamiento

**Universidad Autónoma de Guadalajara** · Ingeniería Biomédica

**Nombre:** _____________________________  **Matrícula:** ______________  **Fecha:** ____________

---

## El caso

Trabajas en el área de análisis de datos de un hospital. El **Servicio de Urgencias** quiere una
herramienta que, con la información disponible en la **primera hora** tras la llegada del paciente
(signos vitales y laboratorios), anticipe si ese paciente **ingresará a la Unidad de
Cuidados Intensivos (UCI) dentro de las primeras 24 horas**. La finalidad es preparar camas y
personal con antelación.

Los datos llegan de **dos hospitales de la red** y en **dos formatos distintos**.

> ⚠️ **Aviso:** los archivos contienen **datos sintéticos** generados con semilla fija a partir de
> rangos clínicos plausibles. **No corresponden a pacientes reales** y ninguna conclusión de esta
> tarea tiene validez clínica.


**Fuera de alcance:** escalamiento, ingeniería de características y codificación one-hot. Son temas
de la **Unidad 4** y no deben aparecer en tu solución.

## Instrucciones

1. Coloca este notebook en la **misma carpeta** que el directorio `Dataset_3/`.
2. Completa **todas** las celdas marcadas con `# TODO`. No borres los comentarios de la consigna.
3. Responde en las celdas marcadas con **✍️** usando texto, no código.
4. **Usa `SEED = 42` en todo el notebook**, salvo donde la consigna pida explícitamente otra semilla.
   Si no fijas la semilla, tus números no coincidirán con los de la revisión.

## Entrega

El notebook **ejecutado de principio a fin** (con las salidas visibles), en formato `.ipynb`,
subido a Moodle. Antes de entregar, ejecuta *Kernel → Restart & Run All* y verifica que no queden
errores.

## Rúbrica

| Criterio | Puntos |
|---|---|
| Parte 1–2 · Ingesta correcta (incluye el manejo de los faltantes), consolidación y limpieza | 20 |
| Parte 3 · `X` / `y` bien construidas y línea base calculada | 20 |
| Parte 4 · División estratificada y evidencia del efecto de `stratify` | 25 |
| Parte 5 · Enfoque (2) aplicado correctamente (la prueba se abre una sola vez) | 25 |
| Parte 6 · Validación cruzada estratificada K = 5 | 10 |
| **Total** | **100** |

---
## Parte 0 · Preparación del entorno *(celda proporcionada, solo ejecútala)*

In [ ]:
# ===== BLOQUE 1 — Librerías y semilla global =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

pd.set_option('display.max_columns', None)

SEED = 42
np.random.seed(SEED)

---
## Parte 1 · Ingesta y consolidación *(arrastre de 3.2)*

| Fuente | Archivo | Formato | Detalle a cuidar |
|---|---|---|---|
| Hospital Norte (general) | `urgencias_hospital_norte.csv` | CSV | Faltantes escritos como `ND` |
| Hospital Sur (centro de referencia) | `urgencias_hospital_sur.xlsx` | Excel | Los registros están en la hoja `registros`; la hoja `diccionario` documenta cada variable |

> 💡 **Antes de escribir código, abre los dos archivos** y mira cómo vienen los datos. Hay un detalle
> en la columna `lactato` que, si lo pasas por alto, hará que toda la columna se cargue como **texto**
> en vez de número. `pd.read_csv` y `pd.read_excel` aceptan el argumento `na_values`.

In [ ]:
# ===== BLOQUE 2 — Lectura de las dos fuentes =====
# TODO 2.1 · Lee el CSV del Hospital Norte en df_csv.
#            Recuerda declarar cómo vienen escritos los faltantes.
df_csv = ...
print(f'CSV   -> {df_csv.shape}')

# TODO 2.2 · Lee la hoja 'registros' del Excel del Hospital Sur en df_xls.
df_xls = ...
print(f'Excel -> {df_xls.shape}')

# TODO 2.3 · Muestra la hoja 'diccionario' para conocer el significado y las unidades
#            de cada variable. (Una sola línea.)


In [ ]:
# ===== BLOQUE 3 — Consolidación y radiografía inicial =====
# TODO 3.1 · Apila las dos fuentes en un solo DataFrame llamado df.
#            Pista: comparten el mismo esquema de columnas.
df = ...
print(f'Dataset consolidado -> {df.shape}\n')

# TODO 3.2 · Verifica el tipo de dato de la columna 'lactato'.
#            Si dice 'object' en lugar de 'float64', regresa al BLOQUE 2: algo falta.
print('Tipo de dato de lactato:', ...)

# TODO 3.3 · Reporta cuántas filas duplicadas hay y cuántos faltantes por columna.
print('Filas duplicadas:       ', ...)
print('Faltantes por columna:')
print(...)

---
## Parte 2 · Limpieza mínima *(arrastre de 3.3)*

Solo cuatro decisiones. Ninguna es nueva: todas se vieron en la sesión de preparación de datos.

| # | Decisión | Motivo |
|---|---|---|
| 1 | Imputar `lactato` con la **mediana** | Distribución sesgada; la mediana no se arrastra con los valores extremos |
| 2 | Eliminar los **duplicados** | Un mismo paciente no puede estar en entrenamiento y en prueba |
| 3 | **Barajar** las filas con semilla fija | Las fuentes llegaron apiladas: primero todo el Norte, después todo el Sur |
| 4 | Mapear `sexo` a **0/1** | Los modelos no operan con texto. Es un **mapeo**, no codificación one-hot |

<br>

> ⚠️ **El orden importa.** Los pasos 1 a 3 deben ocurrir **antes** de cualquier división de los datos.
> Piensa por qué mientras los programas: te lo preguntaré en la reflexión final.

> 💡 Sobre el paso 3: el Hospital Sur es un **centro de referencia** y recibe pacientes más graves.
> Si el dataset queda ordenado por hospital y lo partes tal cual, los subconjuntos no representarán
> a la misma población.

In [ ]:
# ===== BLOQUE 4 — Limpieza =====
# TODO 4.1 · Imputa los faltantes de 'lactato' con la mediana de la columna.
#            Guarda la mediana en una variable e imprímela.
mediana_lactato = ...
df['lactato'] = ...
print(f'Mediana usada para imputar lactato: {mediana_lactato} mmol/L')

# TODO 4.2 · Elimina las filas duplicadas. Imprime cuántas filas quedaron.
print('\nFilas antes: ', len(df))
df = ...
print('Filas después:', len(df))

# TODO 4.3 · Baraja las filas usando random_state=SEED y reinicia el índice.
#            Pista: .sample(frac=1, random_state=...)
df = ...

# TODO 4.4 · Mapea la columna 'sexo': 'Mujer' -> 0, 'Hombre' -> 1.
df['sexo'] = ...

print('\nFaltantes restantes:', df.isna().sum().sum())
df.head()

---
## Parte 3 · Definición del problema, `X` / `y` y **línea base**

> **Pregunta:** ¿este paciente ingresará a la UCI en las primeras 24 horas?

La variable objetivo es `ingreso_uci` y **ya viene registrada** en el expediente: no la derivamos de
ninguna otra columna, así que no hay fuga de datos por construcción. Pero **sí hay una columna que no
debe entrar a `X`**. Identifícala: no es una variable clínica y no aporta nada al modelo.

### La línea base (*baseline*)

Antes de entrenar nada hay que saber **qué tan fácil es el problema**. El modelo más simple posible
—responder siempre la **clase mayoritaria**, sin mirar al paciente— ya acierta en cierta proporción
de los casos. Ese número es la vara con la que se mide todo lo demás: **un modelo que no lo supera
no sirve**, por muy alto que se vea su accuracy.

In [ ]:
# ===== BLOQUE 5 — X, y, distribución de clases y línea base =====
# TODO 5.1 · Construye X (predictoras) y y (objetivo).
#            Deja fuera de X la variable objetivo Y la columna que no es clínica.
X = ...
y = ...

print(f'X (predictoras): {X.shape}')
for c in X.columns:
    print('   -', c)
print(f'y (objetivo):    {y.shape}')

# TODO 5.2 · Imprime la distribución de clases y la proporción de la clase positiva.
print('\nDistribución de clases:')
print(...)
prop_positiva = ...
print(f'\nProporción de la clase positiva (ingreso a UCI): {prop_positiva:.1%}')

# TODO 5.3 · Calcula la LÍNEA BASE: el accuracy de predecir siempre la clase mayoritaria.
linea_base = ...
print(f'LÍNEA BASE: {linea_base:.4f}')

### ✍️ Responde (1–2 líneas)

¿Qué columna dejaste fuera de `X` además de la variable objetivo, y por qué?

> *Tu respuesta:*
>

---
# 3.4 · SEGREGACIÓN DE DATOS

## Parte 4 · Enfoque (1): Entrenamiento + Prueba

| Subconjunto | Proporción | Rol | ¿El modelo lo ve al entrenar? |
|---|---|---|---|
| **Entrenamiento** | 80 % | El modelo aprende de aquí | ✅ Sí |
| **Prueba** | 20 % | Simula pacientes **nuevos** | ❌ No |

Usa `train_test_split` con `test_size=0.20`, `stratify=y` y `random_state=SEED`.

In [ ]:
# ===== BLOQUE 6 — División 80/20 estratificada y primer modelo =====
# TODO 6.1 · Divide X e y en entrenamiento y prueba (80/20, estratificado, semilla SEED).
X_train, X_test, y_train, y_test = ...

print(f'Entrenamiento: {len(X_train)} pacientes ({y_train.mean():.1%} ingresan a UCI)')
print(f'Prueba:        {len(X_test)} pacientes ({y_test.mean():.1%} ingresan a UCI)')

# TODO 6.2 · Entrena una regresión logística SOLO con el conjunto de entrenamiento.
#            Usa LogisticRegression(max_iter=3000).
modelo = ...

# TODO 6.3 · Calcula el accuracy en el conjunto de PRUEBA y compáralo con la línea base
#            de ESE conjunto de prueba.
acc_test = ...
base_test = ...
print(f'\nAccuracy en PRUEBA:  {acc_test:.4f}')
print(f'Línea base:          {base_test:.4f}')
print(f'Ganancia sobre la línea base: {(acc_test - base_test) * 100:+.1f} puntos porcentuales')

### ¿Por qué `stratify=y`?

Con solo ~21 % de casos positivos, un reparto puramente al azar puede dejar la prueba con muy pocos
pacientes que sí ingresaron a UCI. Cuando eso pasa, el accuracy medido ahí dice más sobre el sorteo
que sobre el modelo. Compruébalo comparando ambas versiones sobre varias semillas.

In [ ]:
# ===== BLOQUE 7 — Con estratificación vs. sin estratificación =====
# TODO 7.1 · Para cada semilla de la lista, haz DOS divisiones 80/20: una SIN stratify
#            y otra CON stratify. Imprime la proporción de positivos del conjunto de
#            PRUEBA en cada caso.
print(f'Proporción real de positivos en el dataset: {y.mean():.4f}\n')
print(f'{"semilla":>8} | {"sin stratify":>13} | {"con stratify":>13}')
print('-' * 40)

for rs in [0, 1, 7, 21, 42]:
    # ... completa aquí ...
    pass

### ✍️ Responde (2–4 líneas)

Mira la columna *sin stratify*: ¿entre qué valores se mueve la proporción de positivos de la prueba?
¿Y con `stratify`? Con estos números, ¿dirías que sin estratificar el resultado **siempre sale mal**,
o el problema es otro? Explica.

> *Tu respuesta:*
>
>

---
## Parte 5 · Enfoque (2): Entrenamiento + Validación + Prueba

En cuanto hay **más de un candidato**, el enfoque (1) deja de alcanzar. Si usas la prueba para elegir
al mejor modelo, la prueba deja de ser "datos nunca vistos": aunque el modelo no se entrenó con esas
filas, **tú las miraste para decidir**, y la estimación final queda inflada.

| Subconjunto | Proporción | Rol | ¿Cuántas veces se usa? |
|---|---|---|---|
| **Entrenamiento** | 60 % | El modelo aprende | Una por candidato |
| **Validación** | 20 % | Comparar candidatos | Una por candidato |
| **Prueba** | 20 % | Evaluación **final** | **Una sola vez, al final** |

Se construye en **dos pasos**. Cuidado con el segundo `test_size`: para que la validación sea el 20 %
**del total**, debe ser el 25 % del 80 % restante.

> 🔒 **Regla de oro: la prueba es sagrada.** No la toques hasta el BLOQUE 11.

In [ ]:
# ===== BLOQUE 9 — División en tres subconjuntos =====
# TODO 9.1 · Paso 1: aparta la PRUEBA (20 % del total), estratificada, semilla SEED.
X_temp, X_test2, y_temp, y_test2 = ...

# TODO 9.2 · Paso 2: del 80 % restante, saca la VALIDACIÓN.
#            ¿Qué test_size necesitas para que sea el 20 % del total?
X_train2, X_val, y_train2, y_val = ...

n = len(X)
print(f'Entrenamiento: {len(X_train2):>3} pacientes ({len(X_train2)/n:.0%})')
print(f'Validación:    {len(X_val):>3} pacientes ({len(X_val)/n:.0%})')
print(f'Prueba:        {len(X_test2):>3} pacientes ({len(X_test2)/n:.0%})  <- INTOCABLE hasta el final')

---
## Parte 6 · Validación cruzada estratificada

Las partes 5 dejó el mismo problema en pie: **medimos con un solo corte**, y ese corte es azar.
La validación cruzada lo resuelve rotando todas las divisiones:

1. Se parte el dataset en **K pliegues** (*folds*) del mismo tamaño.
2. Se hacen **K rondas**: en cada una, un pliegue evalúa y los otros K−1 entrenan.
3. Se reportan la **media** (desempeño esperado) y la **desviación estándar** (estabilidad).

En clasificación se usa **`StratifiedKFold`**, por la misma razón por la que usaste `stratify` en la
parte 4.

In [ ]:
# Comparar cuatro candidatos EN VALIDACIÓN
candidatos = {
    'Regresión logística': LogisticRegression(max_iter=3000),
    'Árbol (prof. 3)':     DecisionTreeClassifier(max_depth=3, random_state=SEED),
    'Árbol (prof. 5)':     DecisionTreeClassifier(max_depth=5, random_state=SEED),
    'Árbol (sin límite)':  DecisionTreeClassifier(random_state=SEED),
}

In [ ]:
# ===== BLOQUE 14 — Validación cruzada estratificada, K = 5 =====
# TODO 14.1 · Crea un StratifiedKFold con 5 pliegues, shuffle=True y random_state=SEED.
skf5 = ...

print(f'{"Modelo":<22}{"Media":>9}{"Desv.":>9}   Se reporta como')
print('-' * 62)
scores_por_modelo = {}

# TODO 14.2 · Para cada candidato, obtén sus 5 accuracies con cross_val_score sobre X e y
#             (el dataset COMPLETO) y reporta media y desviación estándar.
for nombre, m in candidatos.items():
    # ... completa aquí ...
    pass

print(f'\nLínea base: {1 - y.mean():.4f}')

In [ ]:
# ===== BLOQUE 15 — Gráfica: accuracy por pliegue (modelo ganador) =====
# TODO 15.1 · Toma del diccionario los 5 accuracies del modelo que ganó en validación.
scores = ...

# --- A partir de aquí el código está completo: solo ejecútalo ---
media, desv = scores.mean(), scores.std()
etiquetas = [f'Pliegue {i}' for i in range(1, len(scores) + 1)]

plt.figure(figsize=(8, 4.5))
plt.grid(axis='y', alpha=0.3, zorder=0)
plt.axhspan(media - desv, media + desv, color='#E8912B', alpha=0.25, zorder=1,
            label=f'Media ± 1 desv. est. ({desv:.4f})')
plt.bar(etiquetas, scores, color='#1E2664', edgecolor='#1E2664', zorder=3)
plt.axhline(media, color='#E8912B', linestyle='--', linewidth=2, zorder=4,
            label=f'Media = {media:.4f}')

for i, a in enumerate(scores):
    plt.text(i, a + 0.005, f'{a:.3f}', ha='center', fontsize=9, color='#1E2664', zorder=5)

plt.ylim(0.70, 0.95)
plt.ylabel('Accuracy')
plt.title('Validación cruzada estratificada (K = 5)')
plt.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()